# DualAMP-Lite · Kaggle 运行（2×T4）

对应计划 V3 的建模主线：ESM-2(t12_35M) 冻结特征 + 多尺度 1D-CNN + 35 维生物物理先验 + Concat 融合，
同源簇分组切分、DeLong/按簇 Bootstrap 统计，最后自动生成 D9–D11 报告。

**运行前检查三件事：**

1. Notebook 右侧 **Accelerator = GPU T4 ×2**（没有 GPU 也能跑，只是慢；`--backend mock` 是纯 CPU 冒烟）；
2. **挂载数据**：把 `raw/` 下的 APD6 / DRAMP / DBAASP / CAMPR4 / UniProt / AMPSphere 文件 + `sources.yaml`
   打成一个私有 Kaggle Dataset，默认路径 `/kaggle/input/dualamp-raw/`；
3. **Internet**：需要用来 `pip install fair-esm` 和首次下载 ESM-2 权重。若关闭 Internet，请把
   `esm2_t12_35M_UR50D.pt` 和本仓库代码一起上传成 Dataset（下面的 cell 会自动识别）。

没有数据时会自动降级为**合成 demo**（合成序列 + mock 编码器），只验证流水线能跑通，指标不可引用。

In [ ]:
import importlib.util, os, shutil, subprocess, sys, textwrap
from pathlib import Path

RANK = None
def pip(*args: str) -> None:
    """Install without letting pip replace the image's CUDA build of torch."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

def run(args: list[str], check: bool = True) -> int:
    cmd = [sys.executable, "-m", "dualamp_lite", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=CODE, check=False)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode

def human(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} GB"

REPO    = Path(os.environ.get("DUALAMP_REPO", "/kaggle/working/leanrea"))
WORK    = Path(os.environ.get("DUALAMP_WORKDIR", "/kaggle/working/DualAMP-Lite"))
RAW     = Path(os.environ.get("DUALAMP_RAW", "/kaggle/input/dualamp-raw"))
WEIGHTS = Path(os.environ.get("DUALAMP_WEIGHTS", "/kaggle/input/dualamp-esm2"))
BRANCH  = os.environ.get("DUALAMP_BRANCH", "arena/4ecb70f1-leanrea")
REPO_URL = "https://github.com/hukeying85-ux/leanrea"

In [ ]:
# --- 依赖（--no-deps 保住镜像自带的 CUDA torch）------------------------------
for module, package in (("Bio", "biopython"), ("yaml", "pyyaml"), ("joblib", "joblib")):
    if importlib.util.find_spec(module) is None:
        pip(package)

if importlib.util.find_spec("esm") is None:
    pip("fair-esm", "--no-deps")          # weights download needs Internet

import numpy, pandas, sklearn, scipy, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU{i}: {torch.cuda.get_device_name(i)} "
              f"{human(torch.cuda.get_device_properties(i).total_memory)}")
print("paths:", REPO, WORK, RAW, sep="\n  ")

In [ ]:
# --- 代码目录：优先用已挂载的仓库，其次 git clone，再其次从 Dataset 拷贝 ------
if not (REPO / "dualamp-lite" / "dualamp_lite").exists():
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=False)
    if not (REPO / "dualamp-lite").exists():
        for candidate in (RAW / "leanrea", RAW / "dualamp-lite", Path("/kaggle/input/leanrea")):
            if candidate.is_dir() and any(candidate.rglob("dualamp_lite/cli.py")):
                if candidate.name == "leanrea":
                    shutil.copytree(candidate, REPO, dirs_exist_ok=True)
                else:  # dataset contains just the package folder
                    (REPO / "dualamp-lite").mkdir(parents=True, exist_ok=True)
                    shutil.copytree(candidate, REPO / "dualamp-lite", dirs_exist_ok=True)
                break

CODE = REPO / "dualamp-lite"
assert (CODE / "dualamp_lite" / "cli.py").exists(), (
    "找不到 dualamp-lite 代码：请开启 Internet 让 git clone 成功，或把仓库作为 Dataset 挂载到 "
    "/kaggle/input/leanrea"
)
WORK.mkdir(parents=True, exist_ok=True)
print("code:", CODE)

In [ ]:
# --- 环境自检：GPU / 依赖 / CD-HIT / 磁盘 --------------------------------------
run(["--workdir", str(WORK), "check"])

In [ ]:
# --- 选择数据源：真实 manifest 存在就用真实数据，否则用合成 demo --------------
manifest = RAW / "sources.yaml"
DEMO = not manifest.exists()
SOURCE_ARGS = ["--demo"] if DEMO else ["--manifest", str(manifest)]
TRAIN_SCALE = ["--seeds", "1,2", "--folds", "3", "--epochs", "8", "--n-boot", "300"] if DEMO \
    else ["--seeds", "1,2,3,4,5", "--folds", "5", "--epochs", "60", "--n-boot", "2000"]
print("=" * 78)
if DEMO:
    print("⚠️  未找到", manifest, "-> 使用合成 demo（只验证流水线，指标不可引用）")
else:
    print("✅ 使用真实数据:", manifest)
print("=" * 78)

## 1) 构建数据集（D2/D3）

去重 → 五层标签审计 → 同源聚类 → 按簇分组切分。`--cluster-method auto` 会优先 CD-HIT，
没有就退到 MMseqs2，再没有就用内置 k-mer 近似（报告里会自动标注这一点）。

In [ ]:
run(["build", *SOURCE_ARGS, "--workdir", str(WORK), "--cluster-method", "auto"])

import json
summary = json.loads((WORK / "dataset" / "dataset_summary.json").read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2)[:2500])

## 2) 提取特征（D4）

ESM-2 第 12 层残基级特征（保留位置信息给 CNN）+ Masked Mean Pooling（不含 BOS/CLS/EOS）+ 35 维理化特征。
Scaler 只在训练集上拟合。已有缓存会自动跳过。

In [ ]:
feat_cmd = ["features", "--workdir", str(WORK), "--batch-size", "64"]
if DEMO:
    feat_cmd += ["--backend", "mock"]
elif WEIGHTS.is_dir() and list(WEIGHTS.glob("*.pt")):
    feat_cmd += ["--backend", "fair-esm", "--weights-dir", str(WEIGHTS), "--local-files-only"]
run(feat_cmd)

## 3) 训练与消融（D5–D9）

先跑便宜的基线（树模型），再跑 ESM-2 系列。中断后重跑会自动复用已完成模型
（`<MODEL>_results.json` + `_scores.npz` 都在且 seeds 一致时跳过）。

In [ ]:
run(["train", "--workdir", str(WORK), "--models", "M0_RF,M0_HGB", *TRAIN_SCALE])

In [ ]:
run(["train", "--workdir", str(WORK), "--models", "M0_CNN_BiGRU,M1,M2", *TRAIN_SCALE])

In [ ]:
run(["train", "--workdir", str(WORK), "--models", "M3,M3_NOPHYS", *TRAIN_SCALE])

M3 与 M3_NOPHYS / M2 的配对比较是**关键消融**：只有当差值区间不跨 0、且超出预设等价界值时，
才能说“理化先验/残基级 CNN 有贡献”。报告会把结论自动写成诚实版本。

选做（仅在 M3 显示增益后）：`--models M4,M5`，或 `--loss focal`（alpha/gamma 在 dev 上选）。

In [ ]:
run(["report", "--workdir", str(WORK)])

import pandas as pd
metrics_path = WORK / "manuscript" / "D10_test_metrics.csv"
metrics = pd.read_csv(metrics_path)
cols = [c for c in ["model", "auroc", "auroc_ci_low", "auroc_ci_high", "auprc", "mcc",
                    "sensitivity", "specificity", "threshold"] if c in metrics.columns]
display(metrics[cols].sort_values("auroc", ascending=False).round(3))

In [ ]:
from IPython.display import Markdown
report = (WORK / "manuscript" / "D10_evaluation_report.md").read_text(encoding="utf-8")
Markdown(report[:6000] + ("\n\n…（完整版见 manuscript/D10_evaluation_report.md）" if len(report) > 6000 else ""))

## 4) 打包产物

`/kaggle/working` 会随 Notebook 的 “Save version” 一起保存；这里另外打一个 zip，
方便直接 `kaggle kernels output` 拉回本地。

In [ ]:
archive = shutil.make_archive("/kaggle/working/dualamp-lite-output", "zip", root_dir=WORK)
print("archive:", archive, human(Path(archive).stat().st_size))
for path in sorted(WORK.rglob("*")):
    if path.is_file() and path.parent.name in {"manuscript", "dataset", "ablation"}:
        print(f"  {path.relative_to(WORK)}  {human(path.stat().st_size)}")

## 5) 时间预算 / 配额

| 步骤 | 8k 条肽、T4 | 说明 |
|---|---|---|
| build | 3–10 min | CD-HIT 0.4/0.8 |
| features | 2–5 min | 8k × ≤50 aa，batch 64 |
| train（7 模型 × 5 seeds × 5 folds） | 60–120 min | 免费额度 30 GPU h/周，够跑 10+ 次 |

## 6) 下一步（不在本 Notebook 内）

* **D11**：把 AMPSphere 100 条体外测试肽（79 阳 + 21 阴）放进 external 层，核对来源后可引用；
* **D12**：PRJNA798058 / PRJNA489760 / PRJNA679346 的样本级元数据核对（`dataset/D1_sample_metadata.csv` 模板已生成）；
* **D13/D14**：reads 回贴 → 原始 integer counts → DESeq2 —— 这是独立的工作量，别把分类分数当表达量。